# Chapter 8.4 - Multi-Branch Networks (GoogLeNet)

GoogLeNet uses Inception blocks: several branches process the same input at different receptive-field scales, then concatenate their output channels. The architecture lesson is that a block can be a small directed computation graph, not only a straight sequence.

## How to use this notebook

Run the notebook from top to bottom in a clean kernel. The code uses small synthetic tensors so that architecture mechanics can be inspected without downloads, `torchvision`, ImageNet-scale images, or long training runs. Before important cells, predict the shape, parameter count, or failure mode, then read the assertions as executable contracts.

## You are done when you can

- explain why an Inception block has multiple branches
- trace branch shapes and channel concatenation
- explain how 1 by 1 bottlenecks reduce computation before larger kernels
- build a tiny GoogLeNet-style classifier
- debug branch concatenation when spatial sizes do not match


In [ ]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_parameters(module):
    return sum(p.numel() for p in module.parameters())

def trace_module_shapes(module, X):
    rows = []
    current = X
    for name, layer in module.named_children():
        current = layer(current)
        rows.append((name, layer.__class__.__name__, shape(current)))
    return rows, current


## 8.4.0 The Problem This Notebook Solves

VGG repeats one simple path. GoogLeNet asks a different question:

```text
what if useful features need different receptive-field sizes at the same network depth?
```

An Inception block answers with branches:

- a 1 by 1 branch
- a 1 by 1 then 3 by 3 branch
- a 1 by 1 then 5 by 5 branch
- a pooling then 1 by 1 branch

The branch outputs are concatenated along the channel dimension. Concatenation means the next layer receives all branch feature maps as one wider tensor.


## 8.4.1 An Inception Block Concatenates Channel Evidence

Each branch must output the same height and width. The channel counts can differ because concatenation happens along the channel axis.

Before running the cell, predict:

- Branch spatial shapes should all be `16 by 16`.
- Output channels should be `4 + 6 + 6 + 4 = 20`.
- The output tensor should have shape `(2, 20, 16, 16)`.


In [ ]:
class Inception(nn.Module):
    def __init__(self, in_channels, c1, c2, c3, c4):
        super().__init__()
        self.b1 = nn.Sequential(nn.Conv2d(in_channels, c1, kernel_size=1), nn.ReLU())
        self.b2 = nn.Sequential(nn.Conv2d(in_channels, c2[0], kernel_size=1), nn.ReLU(),
                                nn.Conv2d(c2[0], c2[1], kernel_size=3, padding=1), nn.ReLU())
        self.b3 = nn.Sequential(nn.Conv2d(in_channels, c3[0], kernel_size=1), nn.ReLU(),
                                nn.Conv2d(c3[0], c3[1], kernel_size=5, padding=2), nn.ReLU())
        self.b4 = nn.Sequential(nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
                                nn.Conv2d(in_channels, c4, kernel_size=1), nn.ReLU())

    def forward(self, X):
        branches = [self.b1(X), self.b2(X), self.b3(X), self.b4(X)]
        self.last_branch_shapes = [shape(branch) for branch in branches]
        return torch.cat(branches, dim=1)


block = Inception(3, c1=4, c2=(4, 6), c3=(4, 6), c4=4)
Y = block(torch.randn(2, 3, 16, 16))

print("branch shapes:", block.last_branch_shapes)
print("output:", shape(Y))
assert shape(Y) == (2, 20, 16, 16)


## 8.4.2 Bottleneck 1 by 1 Convolutions Reduce Large-Kernel Cost

A 5 by 5 convolution from many input channels to many output channels is expensive. Inception often inserts a 1 by 1 convolution first to reduce channel width:

```text
many channels -> fewer channels -> 5 by 5 convolution
```

This is called a bottleneck because information passes through a narrower channel dimension before the expensive operation.


In [ ]:
in_channels = 64
reduced_channels = 16
out_channels = 32

direct_5x5 = in_channels * out_channels * 5 * 5
bottleneck_then_5x5 = in_channels * reduced_channels * 1 * 1
bottleneck_then_5x5 += reduced_channels * out_channels * 5 * 5

print("direct 5x5 weights:", direct_5x5)
print("1x1 bottleneck then 5x5 weights:", bottleneck_then_5x5)

assert bottleneck_then_5x5 < direct_5x5


## 8.4.3 Build a Tiny GoogLeNet-Style Classifier

The full GoogLeNet has many Inception blocks. This notebook uses a tiny version that preserves the design contract:

```text
stem -> Inception block -> pooling -> Inception block -> global average pool -> logits
```

The first Inception block receives 8 channels and outputs 16. After pooling, the second receives 16 and outputs 32.


In [ ]:
googlenet_tiny = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(),
    Inception(8, c1=4, c2=(4, 4), c3=(4, 4), c4=4),
    nn.MaxPool2d(kernel_size=2, stride=2),
    Inception(16, c1=8, c2=(8, 8), c3=(8, 8), c4=8),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(32, 10),
)

X = torch.randn(2, 1, 32, 32)
rows, logits = trace_module_shapes(googlenet_tiny, X)
for row in rows:
    print(row)

assert shape(logits) == (2, 10)


## 8.4.4 Concatenation Preserves Branch Identity as Channels

After concatenation, the next layer sees one tensor. It does not know which branch each channel came from unless you track the channel ranges yourself.

That is the engineering discipline:

```text
branch 1 channels occupy one slice
branch 2 channels occupy the next slice
branch 3 channels occupy the next slice
branch 4 channels occupy the final slice
```

The cell manually verifies the channel slices after concatenation.


In [ ]:
branches = [torch.full((1, 2, 3, 3), fill_value=v) for v in [1.0, 2.0, 3.0]]
merged = torch.cat(branches, dim=1)

print("merged shape:", shape(merged))
print("channel means:", merged.mean(dim=(0, 2, 3)))

assert shape(merged) == (1, 6, 3, 3)
assert torch.equal(merged[:, 0:2], branches[0])
assert torch.equal(merged[:, 2:4], branches[1])
assert torch.equal(merged[:, 4:6], branches[2])


## 8.4.5 Break It Deliberately: Branch Spatial Mismatch

Concatenation along channels only works when every other dimension matches. If one branch changes height or width, `torch.cat(..., dim=1)` rejects the result.

The theory-level mistake is asking the next layer to treat nonaligned spatial grids as if they were feature channels at the same locations.


In [ ]:
class BadInception(nn.Module):
    def __init__(self):
        super().__init__()
        self.good = nn.Conv2d(3, 4, kernel_size=1)
        self.bad = nn.Conv2d(3, 4, kernel_size=3, stride=2, padding=1)

    def forward(self, X):
        return torch.cat([self.good(X), self.bad(X)], dim=1)


try:
    BadInception()(torch.randn(2, 3, 16, 16))
except RuntimeError as error:
    print(type(error).__name__)
    print(str(error).splitlines()[0])
else:
    raise AssertionError("Expected branch spatial mismatch to fail")


## 8.4 Checkpoint

Answer these before moving on. Short markdown answers in the notebook are enough; the checkpoint is meant to test whether you can explain the mechanics without rereading the code.

1. Why does an Inception block use several branches instead of one path?
2. Why must branch outputs agree on height and width before concatenation?
3. How does a 1 by 1 bottleneck reduce the cost of a 5 by 5 branch?
4. After concatenation, where is branch identity stored?
5. Why is an Inception block not naturally represented as a simple `nn.Sequential` inside its own forward logic?
